## 1. Hasta ahora: `os.path`

Construíamos rutas con `os.path.join`, pero el resultado es un `str` y cada operación es una función distinta.

In [33]:
import os

ruta = os.path.join("datos", "smn", "obs_2026-09-27.txt")

print(ruta)

datos/smn/obs_2026-09-27.txt


In [34]:
print(type(ruta))

<class 'str'>


In [35]:
print(os.path.exists(ruta))

True


In [36]:
print(os.path.basename(ruta))

obs_2026-09-27.txt


## 2. `Path`: la ruta como objeto


In [39]:
from pathlib import Path

base = Path("datos")
print(base)

datos


In [41]:
print(type(base))

<class 'pathlib.PosixPath'>


In [47]:
ruta = base.joinpath("smn", "obs_2026-09-27.txt")
#ruta = os.path.join("datos", "smn", "obs_2026-09-27.txt")


print(ruta, type(ruta))

datos/smn/obs_2026-09-27.txt <class 'pathlib.PosixPath'>


In [5]:
print(Path.cwd())        # directorio de trabajo actual
print(ruta.resolve())    # ruta absoluta

/home/atom/Downloads
/home/atom/Downloads/datos/smn/obs_2026-09-27.txt


Otras formas de construir la misma ruta:

In [44]:
#observar, el operador / puede cambiar dependiendo del objeto: 
r2 = Path("datos") / "smn" / "obs_2026-09-27.txt"
# es lo mismo que Path("datos").joinpath("smn", "obs_2026-09-27.txt")

print(r2)

datos/smn/obs_2026-09-27.txt


In [45]:
# mirar estos otros ejemplos
print(3 + 4)
print("ab" + "cd")
print([1, 2] + [3, 4])
print(Path("datos") / "smn")

7
abcd
[1, 2, 3, 4]
datos/smn


In [9]:
# Alcanza con que uno de los dos lados sea un Path
base / "smn"

PosixPath('datos/smn')

## 3. Inspeccionar una ruta

In [48]:
print(ruta)
print(ruta.name)     # nombre completo
print(ruta.stem)     # sin extensión
print(ruta.suffix)   # extensión
print(ruta.parent)   # carpeta que la contiene

datos/smn/obs_2026-09-27.txt
obs_2026-09-27.txt
obs_2026-09-27
.txt
datos/smn


In [49]:
print(ruta.exists())
print(ruta.is_file())
print(ruta.parent.is_dir())

True
True
True


In [50]:
print(Path("no_existe.txt").exists())

False


## 4. Recorrer y buscar archivos

In [53]:
print(base)
list(base.iterdir())

datos


[PosixPath('datos/smn'), PosixPath('datos/leeme.md')]

In [54]:
for elem in base.iterdir():
    print(elem.name, "es directorio?", elem.is_dir())

smn es directorio? True
leeme.md es directorio? False


`glob` busca por patrón en la carpeta; `rglob` también en las subcarpetas:

In [55]:
print("glob:", list(base.glob("*.txt")))

for r in sorted(base.rglob("*.txt")):
    print(r)

glob: []
datos/smn/obs_2026-09-27.txt
datos/smn/obs_2026-09-28.txt
datos/smn/obs_2026-09-29.txt


## 5. Crear carpetas

In [56]:
salida = Path("salida")
salida.exists()

False

In [57]:
salida.mkdir(parents=True, exist_ok=True)   # no falla si ya existe
print( salida.exists(), salida.is_dir() )

True True


---
# Encoding

## 6. Un archivo de texto son bytes

El **encoding** es la tabla que dice qué bytes corresponden a qué caracteres.

In [89]:
texto = "Año áéíóú ⁰˙¿æ¡©" # el número asociado en formato Unicode
[ord(i) for i in texto]

[65, 241, 111, 32, 225, 233, 237, 243, 250, 32, 8304, 729, 191, 230, 161, 169]

In [91]:
chr(241)

'ñ'

In [92]:
texto = "Año áéíóú ⁰˙¿æ¡©"
print(list(texto.encode("utf-8")))    # la ñ ocupa 2 bytes

[65, 195, 177, 111, 32, 195, 161, 195, 169, 195, 173, 195, 179, 195, 186, 32, 226, 129, 176, 203, 153, 194, 191, 195, 166, 194, 161, 194, 169]


In [93]:
print(list(texto.encode("cp1252")))   # la ñ ocupa 1 byte

UnicodeEncodeError: 'charmap' codec can't encode characters in position 10-11: character maps to <undefined>

Los primeros 128 caracteres (**ASCII**: letras sin tildes, dígitos, signos) coinciden en todos: por eso el problema aparece "solo a veces".

In [95]:
print(list("Bariloche".encode("utf-8")))
print(list("Bariloche".encode("cp1252")))

[66, 97, 114, 105, 108, 111, 99, 104, 101]
[66, 97, 114, 105, 108, 111, 99, 104, 101]


In [96]:
print(list("Bariloché".encode("utf-8")))
print(list("Bariloché".encode("cp1252")))

[66, 97, 114, 105, 108, 111, 99, 104, 195, 169]
[66, 97, 114, 105, 108, 111, 99, 104, 233]


## 7. Leer con el encoding equivocado

Los archivos del SMN vienen en **cp1252**. ¿Qué pasa si los leemos como UTF-8?

`open()` acepta un `Path` directamente, no hace falta convertirlo a `str`.

In [97]:
cp = Path("datos/smn/obs_2026-09-28.txt")

with open(cp, "r", encoding="cp1252") as archivo:
    print(archivo.read())

Estación;Fecha;Hora;Estado;Temperatura;Sensación térmica;Humedad;Viento;Presión
San Carlos de Bariloche;28-sep-2026;12:00;Nevadas débiles;1.8;-2.4;91;Oeste 30;1006.5
Ñorquinco;28-sep-2026;12:00;Cubierto;0.4;-4.1;83;Oeste 28;1004.9
Río Gallegos;28-sep-2026;12:00;Algo nublado;6.3;2.0;64;Oeste 40;998.7



In [98]:
with open(cp, "r", encoding="utf-8") as archivo:
    print(archivo.read())

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf3 in position 6: invalid continuation byte

Y al revés: un archivo UTF-8 leído como cp1252 devolverá caracteres incorrectos, pero sin error:

In [99]:
u8 = Path("datos/smn/obs_2026-09-27.txt")

with open(u8, "r", encoding="cp1252") as archivo:
    print(archivo.read())
    

EstaciÃ³n;Fecha;Hora;Estado;Temperatura;SensaciÃ³n tÃ©rmica;Humedad;Viento;PresiÃ³n
San Carlos de Bariloche;27-sep-2026;12:00;Nublado;8.5;No se calcula;75;Oeste 20;1012.3
NeuquÃ©n;27-sep-2026;12:00;Despejado;19.2;No se calcula;38;Sudoeste 15;1015.1
El BolsÃ³n;27-sep-2026;12:00;Llovizna;7.1;4.9;88;Oeste 25;1010.8



In [100]:
with open(u8, "r", encoding="utf-8") as archivo:
    print(archivo.read())

Estación;Fecha;Hora;Estado;Temperatura;Sensación térmica;Humedad;Viento;Presión
San Carlos de Bariloche;27-sep-2026;12:00;Nublado;8.5;No se calcula;75;Oeste 20;1012.3
Neuquén;27-sep-2026;12:00;Despejado;19.2;No se calcula;38;Sudoeste 15;1015.1
El Bolsón;27-sep-2026;12:00;Llovizna;7.1;4.9;88;Oeste 25;1010.8



## 8. Ver los bytes crudos

Si se acuerdan alguna vez vimos la función `repr()` para mostrar caracteres invisibles. Pero si queremos ver byte por byte cómo está codificada la cadena de caracteres la función es `ascii()`


In [102]:
cadena = 'Estación\tBahía Blanca\nTemperatura\t12 ⁰C'
print(cadena)


Estación	Bahía Blanca
Temperatura	12 ⁰C


In [103]:
print(repr(cadena))
          

'Estación\tBahía Blanca\nTemperatura\t12 ⁰C'


In [25]:
print(ascii(cadena))

'Estaci\xf3n\tBah\xeda Blanca\nTemperatura\t12 \u2070C'


In [104]:
# de manera similar el modo `"rb"` al abrir archivos muestra los bytes.
with open(cp, "rb") as f:
    print(f.readline())      # \xf3 es la ó en cp1252

with open(u8, "rb") as f:
    print(f.readline())      # \xc3\xb3 es la ó en UTF-8

b'Estaci\xf3n;Fecha;Hora;Estado;Temperatura;Sensaci\xf3n t\xe9rmica;Humedad;Viento;Presi\xf3n\n'
b'Estaci\xc3\xb3n;Fecha;Hora;Estado;Temperatura;Sensaci\xc3\xb3n t\xc3\xa9rmica;Humedad;Viento;Presi\xc3\xb3n\n'


In [ ]:
with open(u8, "r", encoding="utf-8") as archivo:
    archivo.readline()           # salteamos el encabezado
    linea = archivo.readline()

print(linea)
print(repr(linea))

## 9. ¿Por qué no simplemente no indicar el encoding?

Sin `encoding`, `open()` usa el del sistema operativo: UTF-8 en Linux y macOS, **cp1252 en Windows**. El mismo programa se comporta distinto según la computadora.

In [ ]:
import locale
locale.getpreferredencoding()

### Regla de oro
- **Siempre** indicar `encoding`.
- Al **leer**: el encoding con el que fue escrito el archivo.
- Al **escribir**: `utf-8`.

## 10. Detectar el encoding

Probar UTF-8 primero; si falla, cp1252.

In [ ]:
def leer_texto(ruta):
    """Devuelve (texto, encoding) probando utf-8 y luego cp1252."""
    try:
        with open(ruta, "r", encoding="utf-8") as archivo:
            return archivo.read(), "utf-8"
    except UnicodeDecodeError:
        with open(ruta, "r", encoding="cp1252") as archivo:
            return archivo.read(), "cp1252"


texto, enc = leer_texto(cp)
print(enc)
print(texto)

Que cp1252 puede andar, pero no garantiza que esté bien, hay que mirar el texto.

## 11. Escribir en UTF-8

In [ ]:
copia = salida / "prueba.txt"

with open(copia, "w", encoding="utf-8") as archivo:
    archivo.write("Estación: Ñorquinco, 0.4 °C\n")

with open(copia, "r", encoding="utf-8") as archivo:
    print(archivo.read())

with open(copia, "rb") as archivo:
    print(archivo.read())

---
# Ejercicio

Escribir un programa que:

1. busque todos los `.txt` dentro de `datos/` (incluyendo subcarpetas);
2. los lea, detectando si son `utf-8` o `cp1252`;
3. informe para cada uno: nombre, encoding detectado y cantidad de líneas;
4. cree la carpeta `salida/` si no existe;
5. guarde una copia de cada archivo **en UTF-8** dentro de `salida/`, con el mismo nombre.

**Verificación:** abrir las copias con `encoding="utf-8"` y comprobar que las tildes y eñes se ven bien.

In [ ]:
# Tu solución acá